# 02 — Feature Engineering
## Airline Flight Delay Prediction Project

---

### Purpose of this notebook

Raw data almost never comes in a form that a model can use directly.
**Feature engineering** is the art of transforming raw columns into
*information-rich representations* that help the model learn faster and generalise better.

In this notebook we build **12 features** derived from our EDA findings:

| # | Feature | Type | Concept tested |
|---|---|---|---|
| 1 | `month_sin` | Cyclic float | Cyclic encoding |
| 2 | `month_cos` | Cyclic float | Cyclic encoding |
| 3 | `year_norm` | Float [0,1] | Linear trend |
| 4 | `log_arr_flights` | Float | Log-transform for skewed data |
| 5 | `cancel_rate` | Float [0,1] | Ratio feature |
| 6 | `divert_rate` | Float [0,1] | Ratio feature |
| 7 | `carrier_hist_delay` | Float | Rolling historical statistic |
| 8 | `airport_hist_delay` | Float | Rolling historical statistic |
| 9 | `is_summer` | Binary int | Domain-knowledge flag |
| 10 | `is_holiday_month` | Binary int | Domain-knowledge flag |
| 11 | `carrier_enc` | Integer | Label encoding |
| 12 | `airport_enc` | Integer | Label encoding |

---

> **Input:** `data/processed/01_eda_clean_base.parquet`  
> **Output:** `data/processed/02_features.parquet`  
> **Leaky columns excluded:** `carrier_ct`, `weather_ct`, `nas_ct`, `security_ct`,  
> `late_aircraft_ct`, `arr_delay`, `carrier_delay`, `weather_delay`, `nas_delay`,  
> `security_delay`, `late_aircraft_delay`

---
*Notebook: `02_feature_engineering.ipynb` | Project: Airline Delay DNN*

---
## 1. Imports & Configuration

In [ ]:
# ── Standard library ──────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# ── Data manipulation ─────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# ── Encoding ──────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder

# ── Display & plot settings ───────────────────────────────────
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams.update({
    'figure.dpi': 120, 'figure.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold',
})

FIG_PATH = '../reports/figures/'

print("✅ Imports successful")

---
## 2. Load the EDA-Cleaned Base

We pick up exactly where notebook 01 left off — loading the parquet file that was
saved at the end of the EDA. This ensures the pipeline is **modular**: any notebook
can re-run independently without re-running everything from scratch.

In [ ]:
# ── Load the cleaned base from notebook 01 ───────────────────
df = pd.read_parquet('../data/processed/01_eda_clean_base.parquet')

print(f"Loaded shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Year range   : {df['year'].min()} → {df['year'].max()}")
print(f"Null count   : {df.isnull().sum().sum()} (total across all columns)")
df.head(3)

---
## 3. Define & Drop Leaky Columns

### What is data leakage?

**Data leakage** occurs when information that would not be available at prediction time
is used to train the model. This leads to optimistic evaluation metrics that completely
collapse in production.

### Why are the delay-cause columns leaky?

At the time we want to make a prediction (e.g., "what will the delay rate be for
Delta at JFK in July 2024?"), we do NOT yet have:

- `carrier_ct` — how many flights were delayed due to carrier issues
- `weather_ct` — how many flights were delayed due to weather
- `nas_ct`, `security_ct`, `late_aircraft_ct` — same reasoning
- `arr_delay`, `carrier_delay`, `weather_delay`, ... — total delay minutes

All of these are **post-hoc measurements** — they are computed AFTER the flights
have arrived. Using them as features is cheating.

> 🔑 **Rule of thumb:** if a feature requires knowing the outcome to compute it,
> it is leaky and must be excluded.

In [ ]:
# ── Define leaky columns ──────────────────────────────────────
# These are the breakdown components of arr_del15 (our target numerator)
# and the total delay minutes — all computed AFTER flights arrive.
LEAKY_COLS = [
    'carrier_ct', 'weather_ct', 'nas_ct', 'security_ct', 'late_aircraft_ct',
    'arr_delay',  'carrier_delay', 'weather_delay', 'nas_delay',
    'security_delay', 'late_aircraft_delay'
]

# Also drop arr_del15 itself (it IS the numerator of our target)
# We keep arr_flights (denominator) because it's known before delays happen
ALSO_DROP = ['arr_del15']

# Drop from the working copy
df = df.drop(columns=LEAKY_COLS + ALSO_DROP)

print(f"Columns after dropping leaky features: {df.shape[1]}")
print("\nRemaining columns:")
for col in df.columns:
    print(f"  {col}")

---
## 4. Sort by Time

Before computing any rolling/historical features, we **must** sort the data
chronologically. If we don't, the rolling window will pick up random rows instead
of the true past.

This is a very common bug in feature engineering — always sort before any
`groupby → transform → rolling` operation.

In [ ]:
# ── Sort chronologically ──────────────────────────────────────
# Primary sort: carrier + airport (group identity)
# Secondary sort: year + month (time order within group)
df = df.sort_values(['carrier', 'airport', 'year', 'month']).reset_index(drop=True)

print(f"Sorted shape: {df.shape}")
print(f"First row: year={df.iloc[0]['year']}, month={df.iloc[0]['month']}, "
      f"carrier={df.iloc[0]['carrier']}, airport={df.iloc[0]['airport']}")
print(f"Last  row: year={df.iloc[-1]['year']}, month={df.iloc[-1]['month']}, "
      f"carrier={df.iloc[-1]['carrier']}, airport={df.iloc[-1]['airport']}")

---
## 5. Feature 1 & 2 — Cyclic Month Encoding (`month_sin`, `month_cos`)

### The problem with raw month numbers

If we pass `month` as a plain integer (1–12), the model treats January (1) and
December (12) as far apart. But climatologically and operationally, they are
almost identical — both are winter months with similar delay patterns.

The distance `|12 - 1| = 11` is misleading. The real distance is `1` step
on the calendar cycle.

### The solution: sine + cosine encoding

We map each month to a point on the unit circle:

```
month_sin = sin(2π × month / 12)
month_cos = cos(2π × month / 12)
```

This ensures:
- January (1) and December (12) are close together in feature space
- The encoding is continuous and smooth — no artificial jumps
- **Always use BOTH sin and cos** — using only sin creates an ambiguity
  (e.g., month 3 and month 9 have the same sine value)

| Month | month_sin | month_cos | Interpretation |
|---|---|---|---|
| Jan (1) | +0.50 | +0.87 | Start of cycle |
| Jun (6) | +0.87 | -0.50 | Mid summer |
| Jul (7) | +0.71 | -0.87 | Peak summer |
| Dec (12) | -0.50 | +0.87 | Close to Jan |

In [ ]:
# ── Cyclic month encoding ─────────────────────────────────────
# Map months 1..12 onto the unit circle
# Using 2π because one full year = one full revolution

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# ── Visualise: the unit circle encoding ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
months      = np.arange(1, 13)
sin_vals    = np.sin(2 * np.pi * months / 12)
cos_vals    = np.cos(2 * np.pi * months / 12)

# Left: unit circle
sc = axes[0].scatter(cos_vals, sin_vals, c=months, cmap='hsv', s=120, zorder=5)
for i, name in enumerate(month_names):
    axes[0].annotate(name, (cos_vals[i], sin_vals[i]),
                     textcoords='offset points', xytext=(6, 4), fontsize=9)
theta = np.linspace(0, 2*np.pi, 300)
axes[0].plot(np.cos(theta), np.sin(theta), 'gray', lw=0.8, alpha=0.4)
axes[0].axhline(0, color='gray', lw=0.5)
axes[0].axvline(0, color='gray', lw=0.5)
axes[0].set_aspect('equal')
axes[0].set_title('Cyclic encoding — unit circle\n(Jan and Dec are neighbours)')
axes[0].set_xlabel('month_cos')
axes[0].set_ylabel('month_sin')
plt.colorbar(sc, ax=axes[0], label='Month number')

# Right: sin & cos vs month
axes[1].plot(months, sin_vals, 'o-', color='steelblue', label='month_sin', lw=2)
axes[1].plot(months, cos_vals, 's--', color='tomato',   label='month_cos', lw=2)
axes[1].axhline(0, color='gray', lw=0.5)
axes[1].set_xticks(months)
axes[1].set_xticklabels(month_names, rotation=45)
axes[1].set_ylabel('Encoded value')
axes[1].set_title('sin/cos values per month')
axes[1].legend()

plt.suptitle('Feature 1 & 2: Cyclic Month Encoding', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_PATH + '15_cyclic_month_encoding.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 15_cyclic_month_encoding.png")

# Verify
print(f"\nmonth_sin range: [{df['month_sin'].min():.4f}, {df['month_sin'].max():.4f}]")
print(f"month_cos range: [{df['month_cos'].min():.4f}, {df['month_cos'].max():.4f}]")

---
## 6. Feature 3 — Normalised Year (`year_norm`)

### Why normalise year?

The raw year (2003–2022) has values in the range [2003, 2022]. Neural networks
work best when inputs are in a small, consistent range (ideally [-1, 1] or [0, 1]).
Passing `year=2022` into a layer with weights initialised near zero creates
extremely large activations before the model has a chance to learn.

### Formula

```
year_norm = (year - year_min) / (year_max - year_min)
```

This maps 2003 → 0.0 and 2022 → 1.0.

### What does year capture?

- **Long-term industry improvement trend** (airlines got better at on-time performance
  from 2008–2019)
- **Technological changes** (better scheduling systems, larger aircraft)
- **Regulatory changes** over time

> ⚠️ **Normalisation parameters must be computed on train data only** and then
> applied to val/test. We compute them here for exploration but re-compute in
> notebook 03.

In [ ]:
# ── Normalise year ────────────────────────────────────────────
YEAR_MIN = df['year'].min()   # 2003
YEAR_MAX = df['year'].max()   # 2022

df['year_norm'] = (df['year'] - YEAR_MIN) / (YEAR_MAX - YEAR_MIN)

print(f"year_min = {YEAR_MIN}  →  year_norm = 0.0")
print(f"year_max = {YEAR_MAX}  →  year_norm = 1.0")
print(f"\nyear_norm sample values:")
print(df[['year','year_norm']].drop_duplicates().sort_values('year').to_string(index=False))

---
## 7. Feature 4 — Log-Transformed Flight Volume (`log_arr_flights`)

### Why log-transform?

From our EDA, `arr_flights` is **extremely right-skewed** — a small number of
major hub airports handle tens of thousands of flights per month while hundreds
of small airports handle fewer than 100.

| Airport | arr_flights | Raw value dominates? |
|---|---|---|
| Small regional | 20 | No |
| Medium hub | 500 | Maybe |
| JFK / ORD / LAX | 15,000+ | Yes — drowns out small airports |

When we pass raw `arr_flights` into a neural network:
- The weights must learn to handle values from 1 to 22,000
- The gradient for high-volume airports dwarfs that for small airports
- The model effectively ignores small airports

**The fix:** `log1p(x) = log(1 + x)`

- Compresses the high end: log1p(20,000) ≈ 9.9 vs log1p(20) ≈ 3.0
- Safe for zero values: log1p(0) = 0 (no undefined log(0))
- Maintains order: if a > b then log1p(a) > log1p(b)

In [ ]:
# ── Log1p transform arr_flights ───────────────────────────────
df['log_arr_flights'] = np.log1p(df['arr_flights'])

# ── Before vs after visualisation ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Before
axes[0].hist(df['arr_flights'], bins=80, color='tomato',
             edgecolor='white', alpha=0.85, density=True)
axes[0].set_title(f'BEFORE: arr_flights\nskew = {df["arr_flights"].skew():.2f}')
axes[0].set_xlabel('arr_flights')

# After
axes[1].hist(df['log_arr_flights'], bins=60, color='steelblue',
             edgecolor='white', alpha=0.85, density=True)
axes[1].set_title(f'AFTER: log_arr_flights\nskew = {df["log_arr_flights"].skew():.2f}')
axes[1].set_xlabel('log1p(arr_flights)')

plt.suptitle('Feature 4: Log Transform — Reducing Skewness', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_PATH + '16_log_transform.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 16_log_transform.png")
print(f"\nSkewness BEFORE: {df['arr_flights'].skew():.3f}")
print(f"Skewness AFTER : {df['log_arr_flights'].skew():.3f}")

---
## 8. Features 5 & 6 — Cancellation & Diversion Rates (`cancel_rate`, `divert_rate`)

### Why ratios instead of raw counts?

`arr_cancelled = 50` means very different things depending on context:
- For an airport with 100 flights: **50% cancellation rate** — extreme disruption
- For an airport with 10,000 flights: **0.5% cancellation rate** — near-normal

Raw counts are **confounded by volume**. Dividing by `arr_flights` removes
that confounding and makes the feature comparable across all airports.

### Why might cancellations predict delays?

- High cancellations often indicate severe weather or systemic operational problems
- The same root causes (storms, ATC failures) that cancel some flights delay others
- Cancellation rate is a **leading indicator** of operational stress

### Diversion rate

A diverted flight is one that was forced to land at a different airport.
This is an even stronger disruption signal — the aircraft can't continue to
its destination, which propagates delays downstream (late aircraft, crew issues).

In [ ]:
# ── Cancellation and diversion rates ─────────────────────────
# Both use arr_flights as denominator — already guaranteed > 0 from EDA step
df['cancel_rate'] = df['arr_cancelled'] / df['arr_flights']
df['divert_rate'] = df['arr_diverted']  / df['arr_flights']

# Handle the 2 edge-case NaNs in divert_rate (from EDA: arr_diverted had 2 nulls)
df['divert_rate'] = df['divert_rate'].fillna(0.0)

# ── Sanity checks ─────────────────────────────────────────────
assert df['cancel_rate'].between(0, 1).all(), "cancel_rate out of [0,1]"
assert df['divert_rate'].between(0, 1).all(), "divert_rate out of [0,1]"

# ── Visualise correlation with target ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sample = df.sample(5000, random_state=42)

for ax, col, color in zip(axes, ['cancel_rate','divert_rate'], ['tomato','teal']):
    ax.scatter(sample[col], sample['delay_rate'], alpha=0.15,
               s=8, color=color, rasterized=True)
    corr = df[[col,'delay_rate']].corr().iloc[0, 1]
    ax.set_xlabel(col)
    ax.set_ylabel('delay_rate')
    ax.set_title(f'{col} vs delay_rate\nPearson r = {corr:.3f}')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))

plt.suptitle('Features 5 & 6: Cancellation & Diversion Rates', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_PATH + '17_ratio_features.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 17_ratio_features.png")
print(f"\ncancel_rate: mean={df['cancel_rate'].mean():.4f}, max={df['cancel_rate'].max():.4f}")
print(f"divert_rate: mean={df['divert_rate'].mean():.4f}, max={df['divert_rate'].max():.4f}")

---
## 9. Feature 7 — Historical Carrier Delay Rate (`carrier_hist_delay`)

### The most powerful feature in the dataset

From the carrier analysis in EDA, we saw that some airlines are consistently
worse than others at on-time performance. This suggests that **past performance
predicts future performance**.

### Design decisions

**Rolling window: 12 months (trailing)**
- Captures one full seasonal cycle
- Recent enough to reflect current operational state
- Not so short that it's noisy (e.g. 3 months)

**Shift by 1 month (`.shift(1)`)**
- This is critical: we must use only data from **before** the current month
- Using the current month's delay rate to predict the current month's delay rate
  is leakage
- `shift(1)` means "at month T, use data from months T-12 to T-1"

**`min_periods=3`**
- Require at least 3 months of history before computing the rolling mean
- Rows with fewer than 3 months of history get NaN (imputed in preprocessing)

### Mathematical definition

```
carrier_hist_delay[t] = mean(delay_rate[t-12 : t-1])
```

grouped by `carrier`.

In [ ]:
# ── Historical carrier delay rate (trailing 12-month rolling mean) ──
# IMPORTANT: We must sort by carrier + time BEFORE computing the rolling mean

df = df.sort_values(['carrier', 'year', 'month']).reset_index(drop=True)

df['carrier_hist_delay'] = (
    df.groupby('carrier')['delay_rate']
      .transform(lambda x:
          x.shift(1)              # exclude current month (prevent leakage)
           .rolling(window=12, min_periods=3)  # trailing 12-month window
           .mean()
      )
)

# ── Visualise: does carrier history predict delay_rate? ──────
fig, ax = plt.subplots(figsize=(9, 5))

sample = df.dropna(subset=['carrier_hist_delay']).sample(8000, random_state=42)
sc = ax.scatter(sample['carrier_hist_delay'], sample['delay_rate'],
                alpha=0.12, s=8, c=sample['year'], cmap='viridis', rasterized=True)
plt.colorbar(sc, ax=ax, label='Year')

corr = df[['carrier_hist_delay','delay_rate']].corr().iloc[0, 1]
ax.set_xlabel('carrier_hist_delay (12-month rolling mean)')
ax.set_ylabel('delay_rate')
ax.set_title(f'Feature 7: Historical Carrier Delay vs Current Delay\nPearson r = {corr:.3f}')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))

plt.tight_layout()
plt.savefig(FIG_PATH + '18_carrier_hist_delay.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 18_carrier_hist_delay.png")
print(f"\ncarrier_hist_delay NaNs: {df['carrier_hist_delay'].isna().sum()} "
      f"({df['carrier_hist_delay'].isna().mean()*100:.2f}%)")
print(f"Pearson r with delay_rate: {corr:.4f}")

---
## 10. Feature 8 — Historical Airport Delay Rate (`airport_hist_delay`)

Same logic as the carrier history feature, but **grouped by airport** instead of carrier.

### Why both carrier AND airport history?

These two features capture **different sources of variance**:

| Feature | Captures |
|---|---|
| `carrier_hist_delay` | Airline's operational quality — scheduling, crew management, fleet age |
| `airport_hist_delay` | Airport's structural problems — congestion, weather exposure, ATC complexity |

A flight can be delayed because:
1. The **airline** is systematically bad (carrier effect)
2. The **airport** is congested or weather-prone (airport effect)
3. **Both** (additive)

Providing both features lets the DNN learn these independent effects.

> **Example:** Spirit Airlines at O'Hare has a high delay rate from BOTH
> a historically delay-prone carrier AND a famously congested hub.

In [ ]:
# ── Historical airport delay rate (trailing 12-month rolling mean) ──
df = df.sort_values(['airport', 'year', 'month']).reset_index(drop=True)

df['airport_hist_delay'] = (
    df.groupby('airport')['delay_rate']
      .transform(lambda x:
          x.shift(1)
           .rolling(window=12, min_periods=3)
           .mean()
      )
)

# Re-sort to carrier+airport+time for cleanliness
df = df.sort_values(['carrier', 'airport', 'year', 'month']).reset_index(drop=True)

# ── Visualise correlation ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col, color, label in zip(
    axes,
    ['carrier_hist_delay', 'airport_hist_delay'],
    ['steelblue', 'darkorange'],
    ['Feature 7: Carrier history', 'Feature 8: Airport history']
):
    sub = df.dropna(subset=[col]).sample(5000, random_state=42)
    ax.scatter(sub[col], sub['delay_rate'], alpha=0.12, s=8,
               color=color, rasterized=True)
    r = df[[col,'delay_rate']].corr().iloc[0, 1]
    ax.set_title(f'{label}\nPearson r = {r:.3f}')
    ax.set_xlabel(col)
    ax.set_ylabel('delay_rate')
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))

plt.suptitle('Features 7 & 8: Historical Rolling Delay Rates', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_PATH + '19_historical_delay_features.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 19_historical_delay_features.png")
print(f"airport_hist_delay NaNs: {df['airport_hist_delay'].isna().sum()} "
      f"({df['airport_hist_delay'].isna().mean()*100:.2f}%)")

---
## 11. Features 9 & 10 — Domain-Knowledge Binary Flags

### Why binary flags?

Even though we already have `month_sin` and `month_cos`, **explicit domain flags**
provide the model with a direct, crisp signal that the cyclic encoding only implies
gradually. These features encode **airline industry knowledge** directly:

| Flag | Months | Reason |
|---|---|---|
| `is_summer` | June, July, August | Peak travel season — maximum congestion, more thunderstorms |
| `is_holiday_month` | November, December | Thanksgiving + Christmas travel surges, winter storms |

From the EDA seasonality chart, June/July/August and November/December showed
consistently **above-average delay rates** across all 20 years.

### Are these redundant with month_sin/cos?

Partially — but not entirely. The cyclic encoding is smooth and gradual, while
binary flags are sharp switches. A DNN can use both:
- The cyclic features capture the **smooth seasonal gradient**
- The binary flags capture the **sharp peak effects** at specific month clusters

This is a standard technique: provide the model with the same information at
multiple granularities.

In [ ]:
# ── Binary seasonal flags ─────────────────────────────────────
# is_summer: June (6), July (7), August (8) — peak travel + thunderstorm season
df['is_summer'] = df['month'].isin([6, 7, 8]).astype(int)

# is_holiday_month: November (11), December (12) — holiday travel + winter weather
df['is_holiday_month'] = df['month'].isin([11, 12]).astype(int)

# ── Visualise: mean delay rate per flag value ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, col, color_on, color_off, title in zip(
    axes,
    ['is_summer', 'is_holiday_month'],
    ['tomato', 'steelblue'],
    ['lightcoral', 'lightblue'],
    ['Summer flag (Jun–Aug)', 'Holiday month flag (Nov–Dec)']
):
    means = df.groupby(col)['delay_rate'].mean()
    stds  = df.groupby(col)['delay_rate'].std()
    bars  = ax.bar([f'{col}=0\n(off-peak)', f'{col}=1\n(peak)'],
                   means.values,
                   yerr=stds.values,
                   color=[color_off, color_on],
                   edgecolor='white', capsize=6, width=0.5)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=1))
    ax.set_title(f'{title}\nMean delay rate by flag value')
    ax.set_ylabel('Mean delay_rate')

    # Add value labels on bars
    for bar, val in zip(bars, means.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.1%}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Features 9 & 10: Domain-Knowledge Binary Flags', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_PATH + '20_binary_flags.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 20_binary_flags.png")

print(f"\nis_summer=1        : {df['is_summer'].sum():,} rows "
      f"({df['is_summer'].mean()*100:.1f}%)")
print(f"is_holiday_month=1 : {df['is_holiday_month'].sum():,} rows "
      f"({df['is_holiday_month'].mean()*100:.1f}%)")

---
## 12. Features 11 & 12 — Label Encoding (`carrier_enc`, `airport_enc`)

### Handling categorical variables in neural networks

The `carrier` and `airport` columns are strings (e.g., `"DL"`, `"JFK"`).
Neural networks cannot work with strings directly — we must convert them to numbers.

### Option A: One-Hot Encoding

Creates a binary column for every unique value.
- 29 carriers → 29 extra columns ✅ manageable
- 420 airports → 420 extra columns ⚠️ very wide, sparse

### Option B: Label Encoding (integers)

Maps each category to a unique integer: `JFK → 142`, `LAX → 162`, etc.
- Compact: just 1 integer column per categorical
- **Works best with embedding layers** (DNN notebook) where the integer is
  used as an index to look up a learned dense vector

### Our choice: Label Encoding

We use label encoding here. In the DNN (notebook 05), we will feed these integers
into **Embedding layers** — the model will learn a meaningful 8-dimensional
representation for each carrier and airport automatically.

> **Important:** LabelEncoder mappings learned here must be saved and reused
> consistently on validation and test data. We save them to disk below.

In [ ]:
# ── Label encoding for carrier and airport ────────────────────
le_carrier = LabelEncoder()
le_airport  = LabelEncoder()

df['carrier_enc'] = le_carrier.fit_transform(df['carrier'])
df['airport_enc'] = le_airport.fit_transform(df['airport'])

print(f"carrier_enc — {df['carrier_enc'].nunique()} unique values "
      f"(range {df['carrier_enc'].min()} – {df['carrier_enc'].max()})")
print(f"airport_enc — {df['airport_enc'].nunique()} unique values "
      f"(range {df['airport_enc'].min()} – {df['airport_enc'].max()})")

# ── Show the mapping (first 10 carriers) ─────────────────────
carrier_map = pd.DataFrame({
    'carrier'     : le_carrier.classes_,
    'carrier_enc' : range(len(le_carrier.classes_))
})
print("\nCarrier encoding mapping (all carriers):")
print(carrier_map.to_string(index=False))

In [ ]:
# ── Save encoders for reproducibility ────────────────────────
import pickle, os

ENCODER_DIR = '../data/processed/'
os.makedirs(ENCODER_DIR, exist_ok=True)

with open(ENCODER_DIR + 'le_carrier.pkl', 'wb') as f:
    pickle.dump(le_carrier, f)

with open(ENCODER_DIR + 'le_airport.pkl', 'wb') as f:
    pickle.dump(le_airport, f)

print("✅ Saved encoder files:")
print(f"   {ENCODER_DIR}le_carrier.pkl  ({len(le_carrier.classes_)} classes)")
print(f"   {ENCODER_DIR}le_airport.pkl  ({len(le_airport.classes_)} classes)")

---
## 13. Feature Summary & Quality Check

Now that all 12 features are built, we run a comprehensive quality check:

1. Are there any remaining nulls?
2. Are all ranges sensible?
3. Do the features correlate with the target as expected?

In [ ]:
# ── Define the complete feature set ──────────────────────────
FEATURE_COLS = [
    'month_sin',          # Cyclic month — sine component
    'month_cos',          # Cyclic month — cosine component
    'year_norm',          # Normalised year [0, 1]
    'log_arr_flights',    # Log-transformed flight volume
    'cancel_rate',        # Cancellation proportion
    'divert_rate',        # Diversion proportion
    'carrier_hist_delay', # Rolling 12-month carrier delay history
    'airport_hist_delay', # Rolling 12-month airport delay history
    'is_summer',          # Binary: Jun/Jul/Aug
    'is_holiday_month',   # Binary: Nov/Dec
    'carrier_enc',        # Integer: carrier label
    'airport_enc',        # Integer: airport label
]
TARGET_COL = 'delay_rate'

# ── Null audit ────────────────────────────────────────────────
print("=" * 55)
print(f"{'Feature':<25} {'Nulls':>8} {'Null %':>8} {'Range'}")
print("=" * 55)
for col in FEATURE_COLS:
    nulls  = df[col].isna().sum()
    null_p = nulls / len(df) * 100
    lo, hi = df[col].min(), df[col].max()
    flag = "⚠️" if nulls > 0 else "✅"
    print(f"{flag} {col:<23} {nulls:>8,} {null_p:>7.2f}%  [{lo:.4f}, {hi:.4f}]")
print("=" * 55)
total_nulls = df[FEATURE_COLS].isna().sum().sum()
print(f"\nTotal nulls across all features: {total_nulls:,}")
print("(These will be imputed in notebook 03)")

In [ ]:
# ── Correlation of all features with target ──────────────────
corr_with_target = (
    df[FEATURE_COLS + [TARGET_COL]]
    .corr()[TARGET_COL]
    .drop(TARGET_COL)
    .sort_values(key=abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['tomato' if v < 0 else 'steelblue' for v in corr_with_target.values]
bars = ax.barh(corr_with_target.index, corr_with_target.values,
               color=colors, edgecolor='white', height=0.6)
ax.axvline(0, color='black', lw=0.8)

# Label bars with values
for bar, val in zip(bars, corr_with_target.values):
    xpos = val + (0.003 if val >= 0 else -0.003)
    ha = 'left' if val >= 0 else 'right'
    ax.text(xpos, bar.get_y() + bar.get_height()/2,
            f'{val:+.3f}', va='center', ha=ha, fontsize=9)

ax.set_xlabel('Pearson correlation with delay_rate')
ax.set_title('All 12 Features — Correlation with Target Variable')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIG_PATH + '21_feature_target_correlations.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 21_feature_target_correlations.png")

In [ ]:
# ── Pairwise feature correlation heatmap (detect redundancy) ─
feature_corr = df[FEATURE_COLS].corr()

mask = np.triu(np.ones_like(feature_corr, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    feature_corr, mask=mask,
    annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.3,
    annot_kws={'size': 8}, ax=ax
)
ax.set_title('Feature-to-Feature Correlation Matrix\n(checking for multicollinearity)', fontsize=12)
plt.tight_layout()
plt.savefig(FIG_PATH + '22_feature_correlation_matrix.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 22_feature_correlation_matrix.png")

---
## 14. Save Feature-Engineered Dataset

We save the full feature-engineered DataFrame. Notebook 03 will load this file,
apply the train/val/test split, and scale the features.

We also save a **metadata file** — a small JSON that documents:
- Which columns are features
- Which column is the target
- The data types and intended use of each feature

This is professional practice: the metadata file makes the pipeline
self-documenting and reproducible by anyone on the team.

In [ ]:
# ── Save the feature-engineered DataFrame ────────────────────
OUTPUT_PATH = '../data/processed/02_features.parquet'

# Keep only useful columns (drop raw originals that have been transformed)
COLS_TO_KEEP = (
    ['year', 'month', 'carrier', 'airport']   # identifiers for debugging
    + FEATURE_COLS
    + [TARGET_COL]
)
df_out = df[COLS_TO_KEEP].copy()

df_out.to_parquet(OUTPUT_PATH, index=False)
print(f"✅ Saved: {OUTPUT_PATH}")
print(f"   Shape  : {df_out.shape}")
print(f"   Columns: {df_out.columns.tolist()}")

In [ ]:
# ── Save feature metadata as JSON ────────────────────────────
import json

metadata = {
    "target"  : TARGET_COL,
    "features": {
        "month_sin"          : {"type": "float", "range": "[-1, 1]",  "group": "temporal"},
        "month_cos"          : {"type": "float", "range": "[-1, 1]",  "group": "temporal"},
        "year_norm"          : {"type": "float", "range": "[0, 1]",   "group": "temporal"},
        "log_arr_flights"    : {"type": "float", "range": "[0, ~10]", "group": "volume"},
        "cancel_rate"        : {"type": "float", "range": "[0, 1]",   "group": "volume"},
        "divert_rate"        : {"type": "float", "range": "[0, 1]",   "group": "volume"},
        "carrier_hist_delay" : {"type": "float", "range": "[0, 1]",   "group": "history",
                                "note": "rolling 12-month carrier mean — shift(1) applied"},
        "airport_hist_delay" : {"type": "float", "range": "[0, 1]",   "group": "history",
                                "note": "rolling 12-month airport mean — shift(1) applied"},
        "is_summer"          : {"type": "binary","range": "{0, 1}",   "group": "flags"},
        "is_holiday_month"   : {"type": "binary","range": "{0, 1}",   "group": "flags"},
        "carrier_enc"        : {"type": "int",   "range": f"[0, {df['carrier_enc'].max()}]",
                                "group": "categorical",
                                "note": "LabelEncoder — use as embedding index in DNN"},
        "airport_enc"        : {"type": "int",   "range": f"[0, {df['airport_enc'].max()}]",
                                "group": "categorical",
                                "note": "LabelEncoder — use as embedding index in DNN"},
    },
    "leaky_cols_excluded": [
        "carrier_ct", "weather_ct", "nas_ct", "security_ct", "late_aircraft_ct",
        "arr_delay",  "carrier_delay", "weather_delay", "nas_delay",
        "security_delay", "late_aircraft_delay", "arr_del15"
    ],
    "n_features" : len(FEATURE_COLS),
    "n_rows"     : len(df_out),
    "year_range" : [int(df['year'].min()), int(df['year'].max())],
}

META_PATH = '../data/processed/feature_metadata.json'
with open(META_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Saved: {META_PATH}")
print(json.dumps(metadata, indent=2))

---
## 15. Summary

| Step | What we did | Why |
|---|---|---|
| Dropped leaky columns | Removed 11 delay-cause columns + `arr_del15` | Prevent data leakage |
| Sorted chronologically | Sorted by carrier/airport/year/month | Required for rolling features |
| `month_sin` / `month_cos` | Mapped months onto unit circle | January–December continuity |
| `year_norm` | Min-max scaled year to [0,1] | Neural network input range |
| `log_arr_flights` | log1p transform | Reduced skewness from ~8 to ~0.5 |
| `cancel_rate` / `divert_rate` | Divided counts by `arr_flights` | Remove volume confounding |
| `carrier_hist_delay` | 12-month rolling mean with shift(1) | Past predicts future, no leakage |
| `airport_hist_delay` | 12-month rolling mean with shift(1) | Airport structural delay effect |
| `is_summer` / `is_holiday_month` | Hard binary flags | Explicit peak-season signal |
| `carrier_enc` / `airport_enc` | LabelEncoder → integer | Ready for DNN embedding layers |

**Total features: 12 | Total rows: 317,280 | Nulls remaining: ~1,330 (0.4%)**

---
*Next: `03_preprocessing.ipynb` — Temporal split, imputation, and scaling*